# Train the game's AI on the books (Qwen3-1.7B + LoRA)

**Runtime → Change runtime type → T4 GPU.**

What this does, in order (each step saves to your Google Drive and can be re-run after a disconnect):
1. **Passages**: split your imported books into ~250-word passages.
2. **Teacher**: Qwen3-4B describes each passage the way the game needs it (summary, place, speakers, a second-person version, next choices). *Longest step, roughly 1.5–3 h for all four books on a free T4.*
3. **Stage A**: the model reads all four books (learns the world and the voice). *~20–40 min.*
4. **Stage B**: the model learns the game's own jobs (narrator, ✦ new scenes) from the teacher's examples. *~1–2 h.*
5. **Compare** base vs trained on scenes it never saw.
6. **Export** a GGUF file for Ollama on your Jetson.

Times are estimates; every step has a quick test run first.

**Private data:** your book text and the trained model live only in your Drive. Don't share them or publish the model.

In [ ]:
!pip -q install 'trl>=0.20' peft transformers datasets accelerate
!pip -q uninstall -y torchao  # Colab's old torchao breaks newer peft
import torch; print(torch.cuda.get_device_name(0))

## Get the code
The notebook clones the GitHub repo (push your latest code first: `git push`). If the repo is private, use a URL with a token, or upload a zip made with
`zip -r repo.zip . -x 'packs/*' 'runs/*' '.git/*' '*.zip'` and leave `REPO_URL` empty.

In [ ]:
REPO_URL = "https://github.com/ahmedvictor507/desert-fates.git"
import os
if not os.path.exists('/content/repo'):
    if REPO_URL:
        !git clone -q $REPO_URL /content/repo
    else:
        from google.colab import files
        up = files.upload()
        !unzip -q -o {list(up)[0]} -d /content/repo
%cd /content/repo
!python -m pytest -q tests/test_book_data.py

## Your books
On the Jetson, pack your imported books (only the `chapters.json` files):

```
cd ~/Desktop/Projects/Dune/packs && zip -r ~/books.zip */chapters.json
```

Upload `books.zip` to Google Drive in a folder called **desert-fates**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/desert-fates'
os.makedirs(DRIVE, exist_ok=True)
!unzip -q -o $DRIVE/books.zip -d /content/books
!ls /content/books
!python -m training.book_data passages --books /content/books --out runs/books

## Step 2: Teacher
First a **test on 32 passages**: look at the samples it prints. If they look sensible, run the full pass (remove `--limit`). It skips passages already done, so after a disconnect just re-run.

Optional speed-up (5×+): `!pip -q install vllm` before running; the script uses vLLM automatically if it imports.

In [ ]:
!python -m training.teacher --passages runs/books/passages.jsonl --out $DRIVE/teacher.jsonl --limit 32
import json
rows = [json.loads(l) for l in open(f'{DRIVE}/teacher.jsonl')]
print(f"{sum(bool(r['teacher']) for r in rows)}/{len(rows)} usable")
for r in rows[:2]:
    print(json.dumps(r['teacher'], indent=1, ensure_ascii=False)[:1500], '\n')

In [ ]:
# full teacher pass (resumable)
!python -m training.teacher --passages runs/books/passages.jsonl --out $DRIVE/teacher.jsonl

In [ ]:
!python -m training.book_data tasks --teacher $DRIVE/teacher.jsonl --out runs/books

## Step 3: Stage A, read the books
Test with `--max-steps 20` first (checks memory and speed), then run the full epoch. Watch `loss`: it should fall from ~3 to ~2.x.

In [ ]:
!python -m training.sft_train domain --data runs/books/domain.jsonl --out $DRIVE/lora-books-test --max-steps 20

In [ ]:
# 2 passes over the books (~50 steps each, ~16 min each on a T4); resumable
!python -m training.sft_train domain --data runs/books/domain.jsonl --out $DRIVE/lora-books --epochs 2

## Step 4: Stage B, learn the game's jobs
Continues the stage A adapter. Test first, then the full run.

In [ ]:
!python -m training.sft_train tasks --data runs/books/tasks.jsonl --init $DRIVE/lora-books --out $DRIVE/lora-dune-test --max-steps 20

In [ ]:
!python -m training.sft_train tasks --data runs/books/tasks.jsonl --init $DRIVE/lora-books --out $DRIVE/lora-dune

## Step 5: Base vs trained, on scenes the model never saw

In [ ]:
!python -m training.compare --adapter $DRIVE/lora-dune --data runs/books/holdout.jsonl --n 4

## Step 6: Export for Ollama
Merges the adapter, converts to GGUF (8-bit, ~1.8 GB) and saves it to your Drive.

In [ ]:
!python -m training.export_model --adapter $DRIVE/lora-dune --out /content/dune-merged
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python /content/llama.cpp/convert_hf_to_gguf.py /content/dune-merged --outtype q8_0 --outfile $DRIVE/dune-q8_0.gguf
!ls -lh $DRIVE/dune-q8_0.gguf

## On the Jetson
Download `dune-q8_0.gguf` from your Drive, then:

```
cd ~/Desktop/Projects/Dune
bash training/install_model.sh ~/Downloads/dune-q8_0.gguf dune
python3 play_story.py --gui --pack dune --llm dune
```